# 03. Признаки, baseline и time-series validation

Цель ноутбука: построить лаги, скользящие средние, календарные признаки и проверить простые baseline-подходы на последовательных временных окнах.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.baselines import moving_average_forecast, seasonal_naive
from src.config import PROCESSED_DATA_DIR
from src.features import add_calendar_features, add_lag_rolling_features
from src.metrics import metrics_table
from src.validation import make_time_series_folds, split_by_fold

In [ ]:
daily_sales = pd.read_parquet(PROCESSED_DATA_DIR / 'daily_sales.parquet')
daily_sales['date'] = pd.to_datetime(daily_sales['date'])

features = add_calendar_features(daily_sales)
features = add_lag_rolling_features(features, ['stock_code', 'country'])
features['baseline_seasonal_7'] = seasonal_naive(features, ['stock_code', 'country'])
features['baseline_ma_28'] = moving_average_forecast(features, ['stock_code', 'country'])
features.to_parquet(PROCESSED_DATA_DIR / 'features.parquet', index=False)
features.head()

In [ ]:
folds = make_time_series_folds(features['date'], validation_size=28, n_folds=3, gap=0)
folds

In [ ]:
rows = []
for fold_id, fold in enumerate(folds, start=1):
    _, valid = split_by_fold(features, fold)
    valid = valid.dropna(subset=['baseline_seasonal_7', 'baseline_ma_28'])
    for column in ['baseline_seasonal_7', 'baseline_ma_28']:
        table = metrics_table(valid['sales'], valid[column])
        table['fold'] = fold_id
        table['baseline'] = column
        rows.append(table)

baseline_metrics = pd.concat(rows, ignore_index=True)
baseline_metrics.to_csv(PROCESSED_DATA_DIR / 'baseline_metrics.csv', index=False)
baseline_metrics

## Вывод после запуска

Лучший baseline по WMAPE: `[A]`. Если forecast bias положительный, прогноз завышает спрос; если отрицательный, занижает.